# Silver to gold - Building BI ready fact table

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import IntegerType, FloatType

catalog_name = "edutrack"

In [0]:
df = spark.table(f"{catalog_name}.silver.slv_enrollments")
df.limit(5).display()

### Step 1 - Calculating performance metrics

In [0]:
df = df.withColumn("percentage_score", F.round((F.col("marks_obtained") / F.col("max_marks")) * 100, 2))
df = df.withColumn("total_score", F.round(F.col("marks_obtained") + F.col("assignment_score"), 2))
df = df.withColumn("pass_flag", F.when(F.col("marks_obtained") >= 40, 1).otherwise(0))
df = df.withColumn("date_id", F.date_format(F.col("dt"), "yyyyMMdd").cast(IntegerType()))
df.select("enrollment_id","marks_obtained","assignment_score","percentage_score","total_score","pass_flag").show(5)

### Step 2 - Selecting final columns

In [0]:
gold_df = df.select(
    F.col("date_id"),
    F.col("dt").alias("enrollment_date"),
    F.col("enrollment_id"),
    F.col("student_id"),
    F.col("course_id"),
    F.col("instructor_id"),
    F.col("semester"),
    F.col("status"),
    F.col("marks_obtained"),
    F.col("max_marks"),
    F.col("assignment_score"),
    F.col("total_score"),
    F.col("percentage_score"),
    F.col("attendance_pct"),
    F.col("grade"),
    F.col("pass_flag"),
)
gold_df.limit(5).display()

In [0]:
gold_df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.gold.gld_fact_enrollments")
print("gld_fact_enrollments written:", spark.table(f"{catalog_name}.gold.gld_fact_enrollments").count(), "rows")

### Step 3 - Business queries

In [0]:
spark.sql("""
    SELECT COUNT(*) AS total_enrollments,
           SUM(pass_flag) AS passed,
           ROUND(SUM(pass_flag)*100.0/COUNT(*),1) AS pass_rate_pct,
           ROUND(AVG(percentage_score),1) AS avg_score_pct,
           ROUND(AVG(attendance_pct),1) AS avg_attendance_pct
    FROM edutrack.gold.gld_fact_enrollments
""").show()

In [0]:
spark.sql("""
    SELECT e.course_id, c.course_name, c.department,
           ROUND(AVG(e.percentage_score),1) AS avg_score,
           ROUND(AVG(e.attendance_pct),1) AS avg_attendance,
           COUNT(*) AS enrollments
    FROM edutrack.gold.gld_fact_enrollments e
    JOIN edutrack.gold.gld_dim_courses c ON e.course_id = c.course_id
    GROUP BY e.course_id, c.course_name, c.department
    ORDER BY avg_score DESC
""").show()

In [0]:
spark.sql("""
    SELECT s.department,
           COUNT(*) AS total,
           SUM(e.pass_flag) AS passed,
           ROUND(SUM(e.pass_flag)*100.0/COUNT(*),1) AS pass_rate_pct
    FROM edutrack.gold.gld_fact_enrollments e
    JOIN edutrack.gold.gld_dim_students s ON e.student_id = s.student_id
    GROUP BY s.department
    ORDER BY pass_rate_pct DESC
""").show()

In [0]:
spark.sql("""
    SELECT grade, COUNT(*) AS count,
           ROUND(COUNT(*)*100.0/SUM(COUNT(*)) OVER(),1) AS pct
    FROM edutrack.gold.gld_fact_enrollments
    GROUP BY grade ORDER BY grade
""").show()

In [0]:
spark.sql("""
    SELECT e.student_id, s.name, s.department,
           ROUND(AVG(e.percentage_score),1) AS avg_score,
           ROUND(AVG(e.attendance_pct),1) AS avg_attendance
    FROM edutrack.gold.gld_fact_enrollments e
    JOIN edutrack.gold.gld_dim_students s ON e.student_id = s.student_id
    GROUP BY e.student_id, s.name, s.department
    ORDER BY avg_score DESC LIMIT 5
""").show()